In [1]:
# libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import csv
from copy import deepcopy
from sklearn import model_selection


In [2]:
# Data import
data_dict_int = {}
data_dict = {}
with open('diabetes_binary_5050split_health_indicators_BRFSS2015.csv') as f:
    header = [h for h in next(f).split(",")]
    for name in header:
        data_dict[name] = []
    data_dict_int = deepcopy(data_dict)
    scores = csv.DictReader(f, fieldnames = header)
    for dict in scores:
        for name, value in dict.items():
            data_dict[name].append(value) 
            data_dict_int[name].append(int(float(value)))

df = pd.DataFrame(data_dict_int, columns=header)
display(df.head(6))
display(df.describe())

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income\n
0,0,1,0,1,26,0,0,0,1,0,...,1,0,3,5,30,0,1,4,6,8
1,0,1,1,1,26,1,1,0,0,1,...,1,0,3,0,0,0,1,12,6,8
2,0,0,0,1,26,0,0,0,1,1,...,1,0,1,0,10,0,1,13,6,8
3,0,1,1,1,28,1,0,0,1,1,...,1,0,3,0,3,0,1,11,6,8
4,0,0,0,1,29,1,0,0,1,1,...,1,0,2,0,0,0,0,8,5,8
5,0,0,0,1,18,0,0,0,1,1,...,0,0,2,7,0,0,0,1,4,7


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income\n
count,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,...,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000,70692.000000
mean,0.500000,0.563458,0.525703,0.975259,29.856985,0.475273,0.062171,0.147810,0.703036,0.611795,...,0.954960,0.093914,2.837082,3.752037,5.810417,0.252730,0.456997,8.584055,4.920953,5.698311
std,0.500004,0.495960,0.499342,0.155336,7.113954,0.499392,0.241468,0.354914,0.456924,0.487345,...,0.207394,0.291712,1.113565,8.155627,10.062261,0.434581,0.498151,2.852153,1.029081,2.175196
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,7.000000,4.000000,4.000000
50%,0.500000,1.000000,1.000000,1.000000,29.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,9.000000,5.000000,6.000000
75%,1.000000,1.000000,1.000000,1.000000,33.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,4.000000,2.000000,6.000000,1.000000,1.000000,11.000000,6.000000,8.000000
max,1.000000,1.000000,1.000000,1.000000,98.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000


In [12]:
# get testing and training data
full_0_sample = df.iloc[0:10]
little_sample = df.iloc[35341:35351]
big_sample = df.iloc[24341:46351]

train, test = model_selection.train_test_split(df, shuffle=True)

<figure>
<img src="Capture d’écran 2026-05-10 à 16.57.13.png">
</figure>

In [13]:
def decision_tree(samples, features_list, limit=1000, label="Diabetes_binary", features_used=[]):
    if (entropy(samples) == 0) or (len(samples) <= limit) or (features_used == features_list):
        return end_leaf(samples, label)
    else:
        feature_split, split_gain = best_feature(samples, features_list, features_used)
        if feature_split == None:
            return end_leaf(samples, label)
        print(f"Best feature split is:", feature_split.keys(), "with a gain of", split_gain)
        return {i_feature:decision_tree(i_values[0], features_list, limit, label, i_values[1]) for i_feature, i_values in feature_split.items()} # return {feature_yes: decision_tree(feature_yes_samples, features_lest_yes), feature_no:decision_tree(feature_no_samples, features_less_no)}
        # feature_split = {f"{feature} < {num_threshold}":[left, left_features_list], f"{feature} > {num_threshold}":[right, right_features_list]}

def end_leaf(samples, label="Diabetes_binary", labels_names=["Non-diabetic", "Diabetic"]):
    unique_list = [0, 1]
    end_label = None
    best = 0
    for i in range(len(unique_list)):
        i_count = list(samples[label]).count(i)
        if i_count > best:
            end_label = labels_names[i]
            best = i_count
    return end_label

def best_feature(samples, features_list, features_used):
    best = [None, 0]
    for feature in features_list:
        if feature not in features_used:
            feature_best_split = best_average(feature, samples, features_used)
            if feature_best_split[0] != None and feature_best_split[1] > best[1]:
                best = feature_best_split
                # print(f"Best feature is {feature}'s average:", best[0].keys(), "with a gain of", best[1])
    return best

def best_average(feature, samples, features_used):
    best = [None, 0]
    unique_values = pd.unique(samples[feature].sort_values(axis = 0).reset_index(drop=True))
    if len(unique_values) > 0:
        for i in range(len(unique_values)-1):
            i_average = (unique_values[i] + unique_values[i+1])/2
            children = num_feature_split(feature, i_average, samples, features_used)
            i_gain = information_gain(samples, children)
            if i_gain > best[-1]:
                best = [children, i_gain]
                # print(f"New best average for {feature}:", i_average, "with a gain of", i_gain)
    return best

def entropy(samples, label="Diabetes_binary", unique_values=[0,1]):
    """Compute information entropy of a given node"""
    entropies = []
    total = len(samples)
    for i in unique_values:
        i_counts = list(samples[label]).count(i)
        if i_counts > 0:
            i_proba = i_counts/total
            i_entropy = i_proba * np.log2(1/i_proba)
            entropies.append(i_entropy)
    return np.sum(entropies)

def information_gain(parent, children):
    total = len(parent)
    parent_entropy = entropy(parent)
    weigted_children_entropies = []
    children_information = 0
    for child_sample, child_features_list in children.values():
        child_weigth = len(child_sample)/total
        child_entropy = entropy(child_sample)
        weigted_children_entropies.append(child_weigth*child_entropy)
    for term in weigted_children_entropies:
        children_information += term
    return parent_entropy - children_information

def num_feature_split(feature, num_threshold, samples, features_used):
    left = samples[samples[feature] < num_threshold]
    left_features_list = feature_removal(left, feature, features_used)
    right = samples[samples[feature] > num_threshold]
    right_features_list = feature_removal(right, feature, features_used)
    return {f"{feature} < {num_threshold}":[left, left_features_list], f"{feature} > {num_threshold}":[right, right_features_list]}

def feature_removal(samples, feature, features_used):
    unique_values = pd.unique(samples[feature].sort_values(axis = 0).reset_index(drop=True))
    if len(unique_values) == 1:
        features_used = deepcopy(features_used) + [feature]
    return features_used

features_list = header[1:]
# print(decision_tree(full_0_sample, features_list))
# print(decision_tree(little_sample, features_list))
print(decision_tree(big_sample, features_list))
# print(decision_tree(train, features_list))

Best feature split is: dict_keys(['HighBP < 0.5', 'HighBP > 0.5']) with a gain of 0.10900259242202548
Best feature split is: dict_keys(['GenHlth < 2.5', 'GenHlth > 2.5']) with a gain of 0.08492395279818821
Best feature split is: dict_keys(['Age < 8.5', 'Age > 8.5']) with a gain of 0.0322874886207587
Best feature split is: dict_keys(['BMI < 27.5', 'BMI > 27.5']) with a gain of 0.02393404497490259
Best feature split is: dict_keys(['HighChol < 0.5', 'HighChol > 0.5']) with a gain of 0.013680635931567209
Best feature split is: dict_keys(['GenHlth < 1.5', 'GenHlth > 1.5']) with a gain of 0.005271579223823153
Best feature split is: dict_keys(['Age < 5.5', 'Age > 5.5']) with a gain of 0.004400415510211919
Best feature split is: dict_keys(['GenHlth < 1.5', 'GenHlth > 1.5']) with a gain of 0.021014161733429293
Best feature split is: dict_keys(['BMI < 27.5', 'BMI > 27.5']) with a gain of 0.0286207886891221
Best feature split is: dict_keys(['GenHlth < 1.5', 'GenHlth > 1.5']) with a gain of 0.0154